# Práctica 2 · Valor del dinero en el tiempo aplicado a CETES

**Equipo:** *(escriban aquí el nombre del equipo y sus integrantes)*

Este notebook vive en el repositorio de **tu equipo**, en `notebooks/02_valor_dinero.ipynb`. El enunciado completo está en el repositorio de la clase, en [`practicas/practica_02`](https://github.com/nana-bug/m1-python-finanzas-2026-2/blob/main/practicas/practica_02/README.md).

**Qué se entrega (nivel base)**

1. Las seis funciones en `src/finanzas.py`, cada una con su docstring.
2. Este notebook, que corre de principio a fin y pasa las seis pruebas.
3. La tasa libre de riesgo de su caso, calculada aquí y escrita en el README del equipo.

**Entrega:** viernes 27 de noviembre de 2026, 23:59.

In [ ]:
# CELDA ESTÁNDAR DEL EQUIPO · Córrela siempre primero.
# En Colab descarga el repositorio de TU EQUIPO para poder usar src/finanzas.py.
import os
import subprocess
import sys

USUARIO = "tu-usuario"          # usuario de GitHub dueño del repositorio del equipo
REPO = "tu-repositorio"         # nombre del repositorio del equipo

if "google.colab" in sys.modules:
    if USUARIO == "tu-usuario":
        raise ValueError("Escribe arriba el usuario de GitHub y el nombre del repositorio de tu equipo.")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", f"https://github.com/{USUARIO}/{REPO}.git", f"/content/{REPO}"], check=True)
    else:   # ya estaba descargado: trae los últimos cambios del equipo
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"], check=False)
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy-financial"], check=True)
else:
    while not os.path.isdir("src") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Carpeta de trabajo:", os.getcwd())

## 1 · Sus seis funciones

Esta celda importa las funciones desde `src/finanzas.py` de su repositorio. Si marca `ImportError`, falta alguna función o el nombre no es exactamente el de la tabla del enunciado.

In [ ]:
import math

from src.finanzas import (
    precio_cete,
    rendimiento_cete,
    tasa_efectiva,
    tasa_efectiva_cete,
    valor_futuro,
    valor_presente,
)

## 2 · Las seis pruebas

No modifiquen esta celda. Debe terminar con el mensaje "Las seis pruebas pasan".

Si una prueba falla, Python indica en qué línea. Revisen en este orden: las unidades (¿decimales o porcentaje?), la base (¿360 días?) y el orden de los argumentos.

In [ ]:
assert math.isclose(valor_futuro(100_000, 0.10, 10, m=None), 100_000 * math.e)
assert math.isclose(valor_presente(100, 0.061, 100, m=2), 0.2457, abs_tol=1e-4)
assert math.isclose(tasa_efectiva(0.12, 12), 0.1268, abs_tol=1e-4)
assert round(precio_cete(0.0439, 91), 7) == 9.8902485
assert math.isclose(rendimiento_cete(9.8902485, 91), 0.0439, abs_tol=1e-7)
assert math.isclose(tasa_efectiva_cete(0.0620, 28), 0.0638, abs_tol=1e-4)
print("Las seis pruebas pasan")

## 3 · Aplicación a CETES

Usen los resultados de la **última subasta** de CETES disponible al momento de entregar. Anoten la fecha de la subasta, las cuatro tasas de rendimiento (en decimales) y la fuente de donde las tomaron.

In [ ]:
FECHA_SUBASTA = "10/11/2026"                                        # fecha de la subasta que usan
FUENTE = "Banco de México, resultados de la subasta de valores gubernamentales"
TASAS_CETES = {28: 0.0620, 91: 0.0647, 182: 0.0661, 364: 0.0695}    # plazo en días: tasa anual

**3.1** Para cada plazo calculen el precio del CETE y su tasa efectiva anual, e impriman una tabla. Comprueben también que `rendimiento_cete` regresa la tasa original a partir del precio.

In [ ]:
# Tu código aquí

**3.2 · La tasa libre de riesgo de su caso.** Conviertan la tasa de CETES a 28 días a efectiva anual y después a tasa por día hábil con $(1 + r_{ef})^{1/252} - 1$. Guarden los resultados en `tasa_libre_anual` y `tasa_libre_diaria`.

In [ ]:
# Tu código aquí
print(f"Tasa libre de riesgo efectiva anual: {tasa_libre_anual:.4%}")
print(f"Tasa libre de riesgo por día hábil:  {tasa_libre_diaria:.6%}")

In [ ]:
# Comprobación: la tasa diaria compuesta 252 veces debe regresar la anual
assert math.isclose((1 + tasa_libre_diaria) ** 252 - 1, tasa_libre_anual)
print("Correcto. Copien estas dos cifras, la fecha de la subasta y la fuente al README de su equipo.")

**3.3 · Interpretación.** Con la tasa de hoy, ¿cuánto tendría que subir el precio de su activo en un año solo para empatarle a CETES? ¿Por qué alguien aceptaría invertir en su activo si CETES paga eso sin riesgo? Contesten en tres o cuatro renglones.

*Escribe aquí tu respuesta.*

## 4 · Reto (opcional): TIR con Newton-Raphson

Agreguen a `src/finanzas.py` las funciones `valor_presente_flujos` (la vimos en clase) y `tir_newton(flujos, tiempos=None, r0=0.10, tol=1e-12, max_iter=100)`, que encuentra la tasa interna de retorno con el método de Newton-Raphson:

$$r_{k+1} = r_k - \frac{VP(r_k)}{VP'(r_k)}, \qquad VP'(r) = -\sum_k \frac{t_k\,C_k}{(1 + r)^{t_k + 1}}$$

Después comparen su resultado con `numpy_financial.irr` usando los flujos de abajo. Si no hacen el reto, dejen esta sección como está: el notebook debe correr igual.

In [ ]:
try:
    from src.finanzas import tir_newton, valor_presente_flujos
    hay_reto = True
except ImportError:
    hay_reto = False
    print("Reto no entregado: falta tir_newton o valor_presente_flujos en src/finanzas.py.")

In [ ]:
import numpy_financial as npf

flujos = [-100_000, 25_000, 30_000, 35_000, 40_000]

if hay_reto:
    tir_libreria = npf.irr(flujos)
    print(f"TIR con numpy-financial: {tir_libreria:.8%}")
    # Tu código aquí

**Pregunta del reto.** Prueben `tir_newton` con los flujos `[-100, 230, -132]`, primero con `r0=0.05` y luego con `r0=0.50`. ¿Qué observan y por qué pasa? ¿Qué hace la bisección (`tir`) con esos mismos flujos?

In [ ]:
flujos_raros = [-100, 230, -132]

if hay_reto:
    print("Flujos:", flujos_raros)
    # Tu código aquí

*Escribe aquí tu respuesta.*

## 5 · Antes de entregar

- [ ] **Entorno de ejecución → Reiniciar sesión y ejecutar todo** corre sin errores.
- [ ] Las seis funciones tienen docstring con su fórmula.
- [ ] El README del equipo dice qué tasa libre de riesgo eligieron, de qué subasta y de qué fuente.
- [ ] Si usaron un LLM, lo declararon en el README.
- [ ] El commit de esta semana lo hizo una persona distinta a la de la semana pasada.